# DeepAR — Model Experiment
**Dataset:** Monthly Labor Market (`monthly_labor_market.csv`)  
**Target:** `EMPLOY`  
**Horizon:** 12 months  
**Split:** 80 % train / last 12 months test  
**Metrics:** MAE, MSE  
**Model:** DeepAR (trained from scratch via NeuralForecast)

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv("../data/processed/monthly_labor_market.csv")

df.rename(columns={"date": "ds"}, inplace=True)
df["ds"] = pd.to_datetime(df["ds"], format="%Y:%m")

df = df.dropna(how="all", subset=df.columns[1:])
df = df.ffill()
df = df.dropna()

df.head()

In [ ]:
TARGET = "EMPLOY"

In [ ]:
split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df  = df.iloc[split:split + 12]

y_true = test_df[TARGET].values

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

In [ ]:
# Historical context: full series with major economic events and train/test boundary
EVENTS = {
    "2001-09-01": "9/11 / Dot-com bust",
    "2008-09-01": "Financial crisis",
    "2020-03-01": "COVID-19",
}

y_max = df[TARGET].max()
y_min = df[TARGET].min()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["ds"], df[TARGET], color="steelblue", linewidth=1, label=TARGET)

ax.axvspan(train_df["ds"].iloc[0], train_df["ds"].iloc[-1],
           alpha=0.08, color="green", label="Training period")
ax.axvspan(test_df["ds"].iloc[0], test_df["ds"].iloc[-1],
           alpha=0.3, color="orange", label="Test window (forecast)")

for date, label in EVENTS.items():
    ts = pd.Timestamp(date)
    ax.axvline(ts, color="crimson", linestyle="--", linewidth=1, alpha=0.7)
    ax.text(ts, y_min + (y_max - y_min) * 0.05, label,
            rotation=90, fontsize=8, color="crimson", va="bottom")

ax.set_title("Monthly Employment — Historical Context with Economic Events")
ax.set_xlabel("Date")
ax.set_ylabel("Employment (thousands)")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_nf = train_df[["ds", TARGET]].copy()
df_nf["unique_id"] = "ts1"
df_nf.rename(columns={TARGET: "y"}, inplace=True)

df_nf.head()

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import DeepAR
from neuralforecast.losses.pytorch import DistributionLoss

# scaler_type="standard" normalises the series before training so the
# model does not see raw values in the 100,000s — without this DeepAR
# collapses to near-zero predictions on large-magnitude series.
model = DeepAR(
    h=12,
    input_size=36,
    lstm_n_layers=2,
    lstm_hidden_size=128,
    max_steps=300,
    loss=DistributionLoss(distribution="StudentT"),
    scaler_type="standard",
    learning_rate=1e-3,
    accelerator="cpu",
)

nf = NeuralForecast(models=[model], freq="MS")
nf.fit(df_nf)


In [ ]:
forecast = nf.predict()

# With DistributionLoss the point-forecast column is "DeepAR-median"
pred_col = [c for c in forecast.columns if c.startswith("DeepAR")][0]
deepar_pred = forecast[pred_col].values[-12:]

print("Using column:", pred_col)
print("Shape:       ", deepar_pred.shape)
print("Prediction:  ", deepar_pred)

In [ ]:
deepar_mae = mean_absolute_error(y_true, deepar_pred)
deepar_mse = mean_squared_error(y_true, deepar_pred)
print(f"DeepAR  MAE: {deepar_mae:.2f}")
print(f"DeepAR  MSE: {deepar_mse:.2f}")

In [ ]:
forecast_dates = test_df["ds"].values

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(forecast_dates, y_true,
        color="black", marker="o", label="Actual", linewidth=2)
ax.plot(forecast_dates, deepar_pred,
        linestyle="--", color="steelblue", label="DeepAR", linewidth=1.5)
ax.set_title("DeepAR Forecast vs Actual — Monthly Employment")
ax.set_xlabel("Date")
ax.set_ylabel("Employment (thousands)")
ax.legend()
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\n{'Model':<12} {'MAE':>10} {'MSE':>14}")
print("-" * 38)
print(f"{'DeepAR':<12} {deepar_mae:>10.2f} {deepar_mse:>14.2f}")